# rete-graph — a quick tour

A `.rete` file is a single, immutable, **range-queryable** RDF graph file: host it anywhere that serves HTTP `Range` requests and query it with SPARQL — no server, no database. This notebook tours the Python client:

1. build a small graph (with a Dataset Card, pyramid, and text index),
2. query it — SELECT / ASK / CONSTRUCT, terms, DataFrames,
3. export it to a file and reopen it lazily,
4. open a **remote** 447k-triple graph and watch how little it downloads.

Docs: [Python API](https://caviri.github.io/rete/python.html) · [build tutorial](https://caviri.github.io/rete/python-build-tutorial.html)

In [1]:
# %pip install rete-graph pandas   # <- uncomment on a fresh environment
import rete_graph as rete

rete.__version__

'0.1.0'

## 1. Build a graph with the lazy `Builder`

Every step just records configuration and returns the builder (calls chain); nothing parses or builds until `run()`. Besides raw RDF text you can `add()` an **rdflib `Graph`/`Dataset`** or `add_file("data.ttl")`.

In [2]:
NT = """\
<http://example.org/alice> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://example.org/Person> .
<http://example.org/alice> <http://www.w3.org/2000/01/rdf-schema#label> \"Alice\" .
<http://example.org/alice> <http://example.org/age> \"42\"^^<http://www.w3.org/2001/XMLSchema#integer> .
<http://example.org/bob> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://example.org/Person> .
<http://example.org/bob> <http://www.w3.org/2000/01/rdf-schema#label> \"Bob\"@en .
<http://example.org/bob> <http://example.org/knows> <http://example.org/alice> .
"""

builder = (
    rete.Builder()
    .add(NT)                                  # or .add(rdflib_graph) / .add_file("x.ttl")
    .card(
        title="Tiny people graph",
        description="Two people who know each other — notebook demo.",
        license="CC0-1.0",
        created="2026-07-16",
    )
    .example(                                 # SPARQL examples travel INSIDE the file
        "SELECT ?s ?o WHERE { ?s <http://example.org/knows> ?o }",
        title="Who knows whom?",
        question="Which people know each other?",
    )
    .pyramid(algo="louvain")                  # or "types", or .pyramid(False)
    .text_index()                             # opt-in full-text word index
)

data = builder.run()                          # <- everything happens here
print(f"{len(data):,} bytes", builder.stats)

2,521 bytes {'statements': 6, 'defaultTriples': 6, 'namedGraphs': 0, 'terms': 10, 'pyramidLevels': 1}


## 2. Query

`query()` returns `{variable: Term}` rows for SELECT, a `bool` for ASK, and `(s, p, o)` triples for CONSTRUCT. A `Term` has `.kind`, `.value`, `.datatype`, `.lang`, `.to_python()`, `.n3`.

In [3]:
g = builder.graph()                           # open the built bytes

rows = g.query("""
    SELECT ?person ?label WHERE {
        ?person <http://www.w3.org/2000/01/rdf-schema#label> ?label
    } ORDER BY ?label
""")
for row in rows:
    print(row["person"].value, "->", repr(row["label"].to_python()), f"(lang={row['label'].lang})")

http://example.org/alice -> 'Alice' (lang=None)
http://example.org/bob -> 'Bob' (lang=en)


In [4]:
print("anyone knows anyone?", g.query("ASK { ?s <http://example.org/knows> ?o }"))

(age_row,) = g.query(
    "SELECT ?age WHERE { <http://example.org/alice> <http://example.org/age> ?age }"
)
age = age_row["age"]
print("age:", age.to_python(), type(age.to_python()), "| n3:", age.n3)

anyone knows anyone? True
age: 42 <class 'int'> | n3: "42"^^<http://www.w3.org/2001/XMLSchema#integer>


In [5]:
# SELECT straight into pandas (pip install rete-graph[pandas])
g.query_df("SELECT ?s ?p ?o WHERE { ?s ?p ?o } LIMIT 6")

,s,p,o
0,http://example.org/alice,http://example.org/age,42
1,http://example.org/alice,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/Person
2,http://example.org/alice,http://www.w3.org/2000/01/rdf-schema#label,Alice
3,http://example.org/bob,http://example.org/knows,http://example.org/alice
4,http://example.org/bob,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/Person
5,http://example.org/bob,http://www.w3.org/2000/01/rdf-schema#label,Bob


## 3. Inspect: card, schema, search

The Dataset Card travels **inside** the file — counts and `format_version` were stamped automatically at build time. The schema profile and the searches come from the pyramid and the text index we opted into.

In [6]:
card = g.card()
print({k: card[k] for k in ("title", "license", "quad_count", "term_count", "format_version")})
print("classes:", g.schema()["classes"])
print("text_search('alice'):", g.text_search("alice"))

# The example we embedded rides along in the file — and runs as-is:
(example,) = g.examples()
print("embedded example:", example["title"], "->", g.query(example["sparql"]))

{'title': 'Tiny people graph', 'license': 'CC0-1.0', 'quad_count': 6, 'term_count': 10, 'format_version': 5}
classes: [('http://example.org/Person', 2)]
text_search('alice'): ['http://example.org/alice']
embedded example: Who knows whom? -> [{'s': Term(kind='iri', value='http://example.org/bob', datatype=None, lang=None), 'o': Term(kind='iri', value='http://example.org/alice', datatype=None, lang=None)}]


## 4. Export, reopen lazily

`export()` writes the immutable file. Reopening a path is **lazy** like a remote open — even `card()` fetches only the metadata section's byte range.

In [7]:
import pathlib, tempfile

path = builder.export(pathlib.Path(tempfile.mkdtemp()) / "people.rete")
reopened = rete.open(path)
print(path)
print("same file?", reopened.content_hash() == g.content_hash())
print("card title:", reopened.card()["title"])

/tmp/tmprnlq26vk/people.rete
same file? True
card title: Tiny people graph


## 5. Query a remote graph over HTTP `Range`

The same API against a real dataset: the Spanish state bulletin (BOE) legal graph — **447k triples, 6.9 MB** — hosted on plain object storage. The client fetches only the byte ranges the query touches, and repeated queries reuse the block cache.

In [8]:
boe = rete.open("https://data.graphplaza.com/boe/boe.rete")
print(boe)

for row in boe.query("""
    SELECT ?s ?label WHERE {
        ?s <http://www.w3.org/2000/01/rdf-schema#label> ?label
    } LIMIT 3
"""):
    print("-", row["label"].value[:70])

s = boe.stats()
print(f"fetched {s['bytes']:,} of {s['fileLength']:,} bytes"
      f" ({100 * s['bytes'] / s['fileLength']:.1f}%) in {s['requests']} range requests")

<rete_graph.Graph source='https://data.graphplaza.com/boe/boe.rete' quads=447128>
- , SE AMPLÍA el anexo, en BOE núm. 129 de 31 de mayo de 2017
- , añadiendo anexo 8, en BOE núm. 255, de 22 de octubre de 2008
- , añadiendo anexo, en BOE núm. 92, de 17 de abril de 1990
fetched 1,572,864 of 6,958,628 bytes (22.6%) in 11 range requests


### The dataset ships its own starter queries

CLI-built datasets embed a whole tiered **starter-query library** in their card. `examples()` reads it over one small ranged request — so you can explore a graph you've never seen by running the questions it carries with it.

In [9]:
examples = boe.examples()
print(f"{len(examples)} example queries embedded in the file:\n")
for ex in examples[:5]:
    print(f"  [{ex['tier']:>7}] {ex['title']} — {ex['question']}")

first = examples[0]
print("\nrunning:", first["title"])
for row in boe.query(first["sparql"]):
    print("  ", {k: v.to_python() for k, v in row.items()})

20 example queries embedded in the file:

  [summary] How many statements? — How big is this graph — how many triples?
  [summary] What relationships exist? — Which predicates (relationships) appear in the data?
  [summary] How frequent is each relationship? — How many statements use each predicate?
  [summary] Does the top predicate occur? — Is the most common predicate present at all?
  [  index] A real entity to start from — Give me one concrete entity of the most common type.

running: How many statements?


   {'n': 447128}


## Where next

- **OWL 2 QL reasoning**: `g.query(q, reason=True)` — entailment by query rewriting, works on remote files too.
- **Federation**: `SERVICE <https://query.wikidata.org/sparql> { ... }` inside any query.
- **Your own storage**: `rete.open(reader=obj)` for any object with `read_at(offset, length)` + `len()` (fsspec/S3).
- **Big datasets**: the `rete build` CLI streams and compresses — see the [build tutorial](https://caviri.github.io/rete/python-build-tutorial.html) for the flag mapping.
- Explore published datasets in the [playground](https://caviri.github.io/rete/playground.html).